# 10. Choose the final model

**The rule was written before the neural networks were trained, and it never uses the test photos:**

1. The model must be able to run offline on a phone.
2. The highest **validation AUC** wins.
3. If models are within 0.005 of each other, the one that misses the fewest **severe** validation photos wins.
4. If that is still tied, the smallest model wins.

Test scores are reported afterwards, once the choice has been made. If we chose by test score, that score would look better than it really is.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn

In [ ]:
from pathlib import Path

# Finds the project folder wherever this notebook is opened from.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "DeepDRiD").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the AfriEyeSamsung project folder. Open this notebook from inside it.")
PROCESSED, RESULTS, MODELS = ROOT / "data" / "processed", ROOT / "results", ROOT / "models"
print("Project folder:", ROOT)

In [ ]:
import json

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score

metrics = {f.name.replace("_metrics.json", ""): json.loads(f.read_text()) for f in sorted(RESULTS.glob("*_metrics.json"))}
predictions = {slug: pd.read_csv(RESULTS / f"{slug}_predictions.csv") for slug in metrics}
labels = pd.read_csv(PROCESSED / "clean_labels.csv")
labels["image_size"] = labels.width.astype(int).astype(str) + " x " + labels.height.astype(int).astype(str)
for slug in predictions:
    predictions[slug] = predictions[slug].merge(labels[["image_id", "image_size"]], on="image_id", validate="one_to_one")
print("Models found:", ", ".join(m["model"] for m in metrics.values()))
print("Not run yet:", ", ".join(s for s in ["xgboost", "mobilenet_v3", "efficientnet_b0", "convnext_tiny"] if s not in metrics) or "none")

In [ ]:
import matplotlib.pyplot as plt

BLUE, TEAL, ORANGE, PURPLE, GREY = "#1F4E9E", "#00948F", "#C4581F", "#7A3E9D", "#8A94A6"
INK, MUTED, GRID = "#1B2433", "#566173", "#E3E7F0"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.edgecolor": "#AEB6C5", "axes.labelcolor": MUTED, "axes.titlecolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True,
    "legend.frameon": False,
})

## Compare every model

In [ ]:
rows = []
for slug, m in metrics.items():
    p = predictions[slug]
    v, t = p[p.split == "val"], p[p.split == "test"]
    rows.append({
        "slug": slug, "Model": m["model"], "Type": m.get("kind", "hand-made features"), "Val AUC": m["val_auc"],
        "Val severe missed": int(((v.grade >= 3) & (v.score < m["threshold"])).sum()), "Val severe": int((v.grade >= 3).sum()), "Val false alarms": int(((v.referable == 0) & (v.score >= m["threshold"])).sum()),
        "Test AUC": m["test_auc"], "CI low": m["test_auc_ci"][0], "CI high": m["test_auc_ci"][1], "Test sensitivity": m["test_sensitivity"], "Test specificity": m["test_specificity"],
        "Test severe missed": m["missed_severe_or_worse"], "Size MB": m.get("model_mb", np.nan), "CPU ms": m.get("cpu_ms_per_photo", np.nan), "Phone": m.get("kind") == "neural network",
    })
table = pd.DataFrame(rows).sort_values("Val AUC", ascending=False).reset_index(drop=True)
print(table[["Model", "Type", "Val AUC", "Val severe missed", "Val severe", "Val false alarms", "Size MB", "CPU ms", "Phone"]].to_string(index=False, formatters={"Val AUC": "{:.3f}".format}))

## Apply the rule

Only validation photos are used in this cell.

In [ ]:
candidates = table[table.Phone]
if candidates.empty:
    raise RuntimeError("No phone-ready neural network was found. The small_cnn files should be in the results folder.")
top = candidates["Val AUC"].max()
close = candidates[candidates["Val AUC"] >= top - .005].sort_values(["Val severe missed", "Size MB"])
winner = close.iloc[0]
print("Within 0.005 of the top validation AUC:", ", ".join(close.Model))
print("Chosen model:", winner.Model)

## Now look at the test result

The choice is locked. This chart shows every model's test result, including those that can't run on the phone.

In [ ]:
show = table.sort_values("Test AUC")
ys = np.arange(len(show))
fig, ax = plt.subplots(figsize=(8.4, max(2.8, .55 * len(show) + 1)))
for y, r in zip(ys, show.itertuples()):
    color = TEAL if r.slug == winner.slug else (BLUE if r.Phone else GREY)
    ax.hlines(y, r._10, r._11, color=color, linewidth=3, alpha=.7)
    ax.plot(r._9, y, "o", color=color, markersize=9, markeredgecolor="white")
    ax.text(r._11 + .006, y, f"{r._9:.3f}", va="center", fontsize=9, color=INK)
ax.set_yticks(ys, show.Model); ax.set_xlabel("Test AUC, with 95% interval"); ax.set_title(f"Chosen model: {winner.Model}"); ax.grid(axis="y", visible=False); ax.margins(x=.1, y=.12)
fig.tight_layout(); plt.show()

In [ ]:
bias_file = RESULTS / "camera_bias.csv"
bias = None
if bias_file.exists():
    b = pd.read_csv(bias_file)
    b = b[(b.slug == winner.slug) & (b.Split == "test")]
    bias = None if b.empty else b.iloc[0]

m = metrics[winner.slug]
choice = {
    "slug": winner.slug, "model": m["model"], "val_auc": m["val_auc"], "test_auc": m["test_auc"], "test_auc_ci": m["test_auc_ci"], "test_sensitivity": m["test_sensitivity"], "test_specificity": m["test_specificity"],
    "test_severe_missed": m["missed_severe_or_worse"], "test_severe_total": m["severe_or_worse"], "threshold": m["threshold"],
    "weakest_image_size_auc": None if bias is None else float(bias.weakest), "compared_models": table.Model.tolist(),
    "not_run": [s for s in ["xgboost", "mobilenet_v3", "efficientnet_b0", "convnext_tiny"] if s not in metrics],
}
(RESULTS / "final_choice.json").write_text(json.dumps(choice, indent=1))
print(f"Final model: {choice['model']}")
print(f"Test AUC {choice['test_auc']:.3f} ({choice['test_auc_ci'][0]:.2f} to {choice['test_auc_ci'][1]:.2f}) | Sensitivity {choice['test_sensitivity']:.1%} | Specificity {choice['test_specificity']:.1%}")
print(f"Severe photos missed: {choice['test_severe_missed']} of {choice['test_severe_total']}")
if choice["weakest_image_size_auc"] is not None:
    print(f"AUC in its weakest image size: {choice['weakest_image_size_auc']:.3f}")
if choice["not_run"]:
    print("\nNot compared yet:", ", ".join(choice["not_run"]), "\nThis choice is provisional until those have run.")
print("\nSaved results/final_choice.json")